# {{PROJECT_NAME}} — 03 Analysis

**Stage:** Transformation / feature engineering → Statistical analysis

The hypothesis goes in writing *before* the test runs. Choosing a test after seeing which
one gives the answer you wanted is how an analysis produces a confident number that means
nothing.

> **This notebook is a stage template, not a mandatory step.** Use the notebooks your
> question needs and delete the rest — a descriptive project may never open
> `04_modeling.ipynb`, and a messy dataset may spend most of its life in `01`.
> See `CHECKLIST.md` for the full workflow.

In [ ]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.visualization.theme import PALETTE, save_fig, set_plot_style
set_plot_style()

# Point this at your file in data/raw/. Everything below follows from it.
DATASET = 'REPLACE_ME.csv'

In [ ]:
from src.data.loading import load_raw
from src.data.cleaning import snake_case_columns, strip_whitespace

df = snake_case_columns(strip_whitespace(load_raw(DATASET)))
df.head()

## DECISION 5 — The hypothesis, stated before the test

- **Hypothesis (H1):** _<the specific, falsifiable claim>_
- **Null (H0):** _<what "nothing here" looks like>_
- **Direction:** _<one-sided with a prior reason, or two-sided>_
- **What would falsify it:** _<be specific>_
- **Alpha, and why:** _<0.05 by convention is not a reason; how costly is a false positive here?>_

## DECISION 6 — Which test, and are its assumptions met?

`compare_groups` requires `test=` and never picks for you. From `src.analysis.hypothesis.TESTS`:

| test | Use when | Assumes |
|---|---|---|
| `welch_t` | Two independent groups, comparing means | Roughly normal, or n large enough for the CLT. Unequal variances fine |
| `student_t` | As above | ...and equal variances. Prefer `welch_t` without a reason |
| `mannwhitney` | Two independent groups, skewed or small n | Nothing about normality; tests dominance, not means |
| `paired_t` | The **same** units measured twice | Pairing is real. Wrong for independent groups |

- **Test:** _<name>_
- **Assumptions checked how:** _<the histograms in 02, group sizes, a normality check>_
- **If an assumption fails:** _<fallback, e.g. mannwhitney>_

In [ ]:
from src.analysis.hypothesis import TESTS, compare_groups

for name, description in TESTS.items():
    print(f'{name}:\n  {description}\n')

In [ ]:
# Fill from DECISION 5 and 6, then run.
VALUE_COL = None
GROUP_COL = None
TEST      = None

if all([VALUE_COL, GROUP_COL, TEST]):
    result = compare_groups(df, VALUE_COL, GROUP_COL, test=TEST)
else:
    print('Answer DECISION 5 and 6 first, then set VALUE_COL, GROUP_COL and TEST.')

### Reading

_<State the effect size and its direction in plain words before the p-value. Is the
difference large enough to matter for the decision this analysis informs? A significant
result with a trivial effect is a true statement about an unimportant thing.>_

In [ ]:
from src.analysis.hypothesis import chi2_independence

# For two categorical variables. Watch the expected-count warning on small samples.
CAT_A, CAT_B = None, None

if CAT_A and CAT_B:
    chi2_independence(df, CAT_A, CAT_B)
else:
    print('Set CAT_A and CAT_B if you are testing two categorical variables.')

## DECISION 7 — Is any transformation or derived feature justified?

Skip this cell entirely if the raw columns answer the question — most of the time they do.

- **Feature:** _<name and formula>_
- **Why the question needs it:** _<not "it improved the score">_
- **What it assumes:** _<e.g. a log transform assumes multiplicative effects>_

In [ ]:
# Derived features, if DECISION 7 justified any. Keep the formula next to the reason.
# df['income_per_person'] = df['household_income'] / df['household_size']

## Findings

State each claim with the number that supports it, then say what would have to be true for
the claim to be wrong.

- **Claim:** _<...>_ — **evidence:** _<statistic, effect size, n>_
- **Confidence:** _<high / medium / low>_ because _<...>_
- **What this does not show:** _<the causal reading you are NOT making, and why>_

**Next stage:** _<04_modeling if DECISION 8 justifies it / straight to reporting>_